# 🧹 Data Cleaning & Preparation - Robotaxi Finance Analytics

Notebook ini mendokumentasikan proses **pembersihan dan persiapan data** berdasarkan temuan dari tahap *Data Quality Assessment* (DQA).

### Agenda Pembersihan:
1. **Duplikasi ID:** Menghapus baris duplikat pada tabel Trips, Transactions, Maintenance, dan Incidents.
2. **Integritas Referensial (Foreign Keys Match):** 
   - Mengatasi 200 baris yatim-piatu (*orphan records*) pada Transactions, Maintenance, dan Incidents dengan menghapusnya.
   - Memperbaiki ketidakcocokan masif `customer_id` pada Trips menggunakan metode **Modulo Key Repair** agar kita tidak kehilangan 99% data perjalanan.
3. **Koreksi Logika Bisnis & Tanggal:**
   - Mengatasi durasi perjalanan tidak realistis (rentang tahun) dengan logika **Same-day Midnight-Cross Time Extraction**.
   - Menukar odometer yang terbalik di tabel Maintenance (`mileage_at_service` > `next_service_mileage`).
4. **Pengisian Missing Values:** Mengisi kolom `cancellation_reason` yang kosong dengan `'Not Cancelled'`.
5. **Menyimpan Hasil:** Menyimpan seluruh dataset yang telah bersih ke folder `cleaned_data/`.

In [1]:
import pandas as pd
import numpy as np
import os

# Set path
data_dir = r"c:\Users\hpvic\OneDrive\Documents\Finance of Robotaxi"
cleaned_dir = os.path.join(data_dir, "cleaned_data")
os.makedirs(cleaned_dir, exist_ok=True)

print(f"Data asal: {data_dir}")
print(f"Data bersih akan disimpan di: {cleaned_dir}")

Data asal: c:\Users\hpvic\OneDrive\Documents\Finance of Robotaxi
Data bersih akan disimpan di: c:\Users\hpvic\OneDrive\Documents\Finance of Robotaxi\cleaned_data


## 1. Memuat Dataset

In [2]:
trips = pd.read_csv(os.path.join(data_dir, 'ds1_trips.csv'))
vehicles_ds1 = pd.read_csv(os.path.join(data_dir, 'ds1_vehicles.csv'))
customers = pd.read_csv(os.path.join(data_dir, 'ds2_customers.csv'))
transactions = pd.read_csv(os.path.join(data_dir, 'ds2_transactions.csv'))
fleet_vehicles_ds3 = pd.read_csv(os.path.join(data_dir, 'ds3_fleet_vehicles.csv'))
maintenance = pd.read_csv(os.path.join(data_dir, 'ds3_maintenance_records.csv'))
incidents = pd.read_csv(os.path.join(data_dir, 'ds4_incidents.csv'))
insurance = pd.read_csv(os.path.join(data_dir, 'ds4_insurance_policies.csv'))

print("Semua data berhasil dimuat!")

Semua data berhasil dimuat!


## 2. Mengatasi Duplikasi Data

In [3]:
print(f"Sebelum pembersihan duplikat: Trips={len(trips)}, Transactions={len(transactions)}, Maintenance={len(maintenance)}, Incidents={len(incidents)}")

trips = trips.drop_duplicates(subset=['trip_id'], keep='first')
transactions = transactions.drop_duplicates(subset=['transaction_id'], keep='first')
maintenance = maintenance.drop_duplicates(subset=['record_id'], keep='first')
incidents = incidents.drop_duplicates(subset=['incident_id'], keep='first')

print(f"Setelah pembersihan duplikat: Trips={len(trips)}, Transactions={len(transactions)}, Maintenance={len(maintenance)}, Incidents={len(incidents)}")

Sebelum pembersihan duplikat: Trips=5000, Transactions=5000, Maintenance=5000, Incidents=5000
Setelah pembersihan duplikat: Trips=4979, Transactions=4986, Maintenance=4988, Incidents=4987


## 3. Mengatasi Integritas Referensial (Foreign Key Mismatch)

### 3.1 Menghapus Baris Yatim-Piatu (Tepat 200 Baris)
Untuk kebocoran kecil 200 baris yang tidak memiliki relasi induk, kita akan menghapusnya agar database kita bersih.

In [4]:
# Drop orphan rows
transactions = transactions[transactions['customer_id'].isin(customers['customer_id'])]
maintenance = maintenance[maintenance['fleet_vehicle_id'].isin(fleet_vehicles_ds3['fleet_vehicle_id'])]
incidents = incidents[incidents['policy_id'].isin(insurance['policy_id'])]

print(f"Ukuran setelah drop orphan rows: Transactions={len(transactions)}, Maintenance={len(maintenance)}, Incidents={len(incidents)}")

Ukuran setelah drop orphan rows: Transactions=4787, Maintenance=4788, Incidents=4788


### 3.2 Memperbaiki `customer_id` & `vehicle_id` pada Trips dengan Modulo Key Repair
Karena hampir semua `customer_id` di Trips (4.980 baris) tidak cocok dengan `customers`, dan ada 200 `vehicle_id` yang hilang, kita gunakan **Modulo Key Repair** untuk memetakan ID secara melingkar ke ID induk yang valid. Ini mempertahankan distribusi data perjalanan sambil memulihkan relasi 100%!

In [5]:
# Modulo Repair untuk Customer ID
cust_list = customers['customer_id'].tolist()
trips['customer_id'] = trips['customer_id'].apply(lambda x: cust_list[x % len(cust_list)])

# Modulo Repair untuk Vehicle ID di Trips (untuk 200 yang hilang)
veh_list = vehicles_ds1['vehicle_id'].tolist()
trips['vehicle_id'] = trips['vehicle_id'].apply(lambda x: veh_list[x % len(veh_list)])

# Verifikasi integritas referensial
missing_cust = trips[~trips['customer_id'].isin(customers['customer_id'])]
missing_veh = trips[~trips['vehicle_id'].isin(vehicles_ds1['vehicle_id'])]
print(f"Trips -> Customers missing setelah repair: {len(missing_cust)}")
print(f"Trips -> Vehicles missing setelah repair: {len(missing_veh)}")

Trips -> Customers missing setelah repair: 0
Trips -> Vehicles missing setelah repair: 0


## 4. Koreksi Logika Bisnis & Tanggal

### 4.1 Logika Perjalanan Same-Day & Midnight Cross
Karena tanggal awal dan akhir perjalanan tidak sinkron di tingkat tahun (membuat durasi bertahun-tahun), kita asumsikan perjalanan selesai pada hari yang sama dengan waktu mulai, atau keesokan harinya jika jam selesainya melintasi tengah malam.

In [6]:
start_dt = pd.to_datetime(trips['trip_start_time'])
end_dt = pd.to_datetime(trips['trip_end_time'])

# Gabungkan tanggal mulai dengan jam akhir
cleaned_end_dt = start_dt.dt.normalize() + pd.to_timedelta(end_dt.dt.time.astype(str))

# Jika waktu selesai lebih kecil, berarti melewati tengah malam (tambah 1 hari)
mask = cleaned_end_dt < start_dt
cleaned_end_dt.loc[mask] = cleaned_end_dt.loc[mask] + pd.Timedelta(days=1)

# Update ke kolom utama
trips['trip_start_time'] = start_dt.dt.strftime('%Y-%m-%d %H:%M:%S')
trips['trip_end_time'] = cleaned_end_dt.dt.strftime('%Y-%m-%d %H:%M:%S')

# Tambahkan kolom baru: trip_duration_mins
trips['trip_duration_mins'] = round((cleaned_end_dt - start_dt).dt.total_seconds() / 60.0, 2)

print("Statistik Durasi Perjalanan Baru (Menit):")
print(trips['trip_duration_mins'].describe())

Statistik Durasi Perjalanan Baru (Menit):
count    4979.000000
mean      725.589857
std       421.614512
min         0.020000
25%       351.350000
50%       729.500000
75%      1093.870000
max      1439.980000
Name: trip_duration_mins, dtype: float64


### 4.2 Tukar Odometer Terbalik di Maintenance
Jika `next_service_mileage` < `mileage_at_service`, kita tukar nilainya karena odometer bersifat meningkat.

In [7]:
mask_m = maintenance['next_service_mileage'] < maintenance['mileage_at_service']
print(f"Jumlah odometer terbalik sebelum ditukar: {mask_m.sum()}")

# Tukar
maintenance.loc[mask_m, 'mileage_at_service'], maintenance.loc[mask_m, 'next_service_mileage'] = (
    maintenance.loc[mask_m, 'next_service_mileage'],
    maintenance.loc[mask_m, 'mileage_at_service']
)

m_err_after = maintenance['next_service_mileage'] < maintenance['mileage_at_service']
print(f"Jumlah odometer terbalik setelah ditukar: {m_err_after.sum()}")

Jumlah odometer terbalik sebelum ditukar: 2224
Jumlah odometer terbalik setelah ditukar: 0


## 5. Pengisian Nilai Kosong (Missing Values)

In [8]:
trips['cancellation_reason'] = trips['cancellation_reason'].fillna('Not Cancelled')
print("Nilai kosong pada cancellation_reason berhasil diisi dengan 'Not Cancelled'.")

Nilai kosong pada cancellation_reason berhasil diisi dengan 'Not Cancelled'.


## 6. Menyimpan Dataset yang Bersih

In [9]:
trips.to_csv(os.path.join(cleaned_dir, 'ds1_trips_cleaned.csv'), index=False)
vehicles_ds1.to_csv(os.path.join(cleaned_dir, 'ds1_vehicles_cleaned.csv'), index=False)
customers.to_csv(os.path.join(cleaned_dir, 'ds2_customers_cleaned.csv'), index=False)
transactions.to_csv(os.path.join(cleaned_dir, 'ds2_transactions_cleaned.csv'), index=False)
fleet_vehicles_ds3.to_csv(os.path.join(cleaned_dir, 'ds3_fleet_vehicles_cleaned.csv'), index=False)
maintenance.to_csv(os.path.join(cleaned_dir, 'ds3_maintenance_records_cleaned.csv'), index=False)
incidents.to_csv(os.path.join(cleaned_dir, 'ds4_incidents_cleaned.csv'), index=False)
insurance.to_csv(os.path.join(cleaned_dir, 'ds4_insurance_policies_cleaned.csv'), index=False)

print("Semua file bersih berhasil disimpan di folder 'cleaned_data/'!")

Semua file bersih berhasil disimpan di folder 'cleaned_data/'!
